# ASL-Citizens Data Pipeline
Downloads ASL-Citizens videos for our 50 target classes, extracts MediaPipe landmarks,
and saves .npy files to Google Drive. Raw videos are deleted after each batch to stay within storage limits.

**Run order:** top to bottom. T4 GPU recommended (Runtime → Change runtime type → T4 GPU).

In [ ]:
# Cell 1 — Install dependencies
!pip install mediapipe datasets huggingface_hub tqdm --quiet

In [ ]:
# Cell 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3 — Clone repo and copy existing labels from Drive
import os, shutil

if not os.path.exists('/content/SignSync'):
    !git clone https://github.com/HaideyAli/SignSync.git /content/SignSync

os.makedirs('/content/SignSync/data', exist_ok=True)

shutil.copy('/content/drive/MyDrive/SignBridge/data/labels_50.json',
            '/content/SignSync/data/labels_50.json')
print('labels_50.json copied.')

In [ ]:
# Cell 4 — Restore existing WLASL landmarks from Drive (so we don't lose them)
import zipfile, glob

os.makedirs('/content/SignSync/data/landmarks', exist_ok=True)
zip_path = '/content/drive/MyDrive/SignBridge/data/landmarks.zip'

if os.path.exists(zip_path):
    print('Unzipping existing WLASL landmarks...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/SignSync/data/landmarks')
    count = len(glob.glob('/content/SignSync/data/landmarks/*.npy'))
    print(f'Restored {count} existing landmark files.')
else:
    print('No existing landmarks.zip found — starting fresh.')

In [ ]:
# Cell 5 — Load our 50 target labels
import json

with open('/content/SignSync/data/labels_50.json', encoding='utf-8') as f:
    label_map = json.load(f)   # {word: idx}

TARGET_WORDS = set(w.lower() for w in label_map.keys())
print(f'Target classes ({len(TARGET_WORDS)}): {sorted(TARGET_WORDS)}')

In [ ]:
# Cell 6 — Scan ASL-Citizens metadata to find matching labels (no video download yet)
from datasets import load_dataset

print('Streaming ASL-Citizens metadata (no videos downloaded yet)...')
ds = load_dataset('drdanjwalker/asl-citizens', split='train', streaming=True)

matched_samples = []   # list of {label, video_path_or_id} dicts
seen_labels = set()

for sample in ds:
    word = sample['label'].strip().lower()
    if word in TARGET_WORDS:
        matched_samples.append(sample)
        seen_labels.add(word)

unmatched = TARGET_WORDS - seen_labels
print(f'Matched {len(seen_labels)}/50 classes, {len(matched_samples)} total samples')
print(f'Unmatched labels (will use WLASL data only): {sorted(unmatched)}')

In [ ]:
# Cell 7 — MediaPipe setup
import urllib.request
from pathlib import Path

MODEL_URL = (
    'https://storage.googleapis.com/mediapipe-models/'
    'holistic_landmarker/holistic_landmarker/float16/1/holistic_landmarker.task'
)
MODEL_PATH = Path('/content/holistic_landmarker.task')

if not MODEL_PATH.exists():
    print('Downloading MediaPipe model...')
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print('Done.')

import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
import numpy as np, cv2

NUM_VALUES = 258

def landmarks_from_frame(result):
    def hand(lms):
        if lms:
            return np.array([[lm.x, lm.y, lm.z] for lm in lms], dtype=np.float32).flatten()
        return np.zeros(63, dtype=np.float32)
    def pose(lms):
        if lms:
            return np.array([[lm.x, lm.y, lm.z, lm.visibility or 0.0] for lm in lms], dtype=np.float32).flatten()
        return np.zeros(132, dtype=np.float32)
    return np.concatenate([hand(result.left_hand_landmarks),
                           hand(result.right_hand_landmarks),
                           pose(result.pose_landmarks)])

def process_video(video_path, detector):
    cap = cv2.VideoCapture(str(video_path))
    frames = []
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, (640, 480))
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        frames.append(landmarks_from_frame(detector.detect(mp_img)))
    cap.release()
    return np.array(frames, dtype=np.float32) if frames else np.zeros((1, NUM_VALUES), dtype=np.float32)

opts = mp_vision.HolisticLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=str(MODEL_PATH)),
    running_mode=mp_vision.RunningMode.IMAGE,
)
print('MediaPipe ready.')

In [ ]:
# Cell 8 — Extract landmarks from ASL-Citizens (batch by batch, delete videos after each)
# Saves .npy files directly to /content/SignSync/data/landmarks/
# Skips any file that already exists — safe to re-run after disconnection

import tempfile, os
from tqdm import tqdm

LANDMARKS_DIR = Path('/content/SignSync/data/landmarks')
LANDMARKS_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 200   # process 200 videos, then save checkpoint to Drive

saved, skipped, errors = 0, 0, 0

with mp_vision.HolisticLandmarker.create_from_options(opts) as detector:
    batch = []
    for i, sample in enumerate(tqdm(matched_samples, desc='Processing')):
        word = sample['label'].strip().lower()
        # Use index as unique ID within ASL-Citizens
        out_path = LANDMARKS_DIR / f'{word}_ac{i:05d}.npy'

        if out_path.exists():
            skipped += 1
            continue

        try:
            # sample['video'] is a dict with 'path' key pointing to a local temp file
            # The datasets library downloads it on access
            video_path = sample['video']['path']
            seq = process_video(video_path, detector)
            np.save(out_path, seq)
            saved += 1
            batch.append(str(out_path))
        except Exception as e:
            print(f'Error on sample {i}: {e}')
            errors += 1
            continue

        # Every BATCH_SIZE files, checkpoint to Drive
        if len(batch) >= BATCH_SIZE:
            total = len(list(LANDMARKS_DIR.glob('*.npy')))
            print(f'\nCheckpoint: {total} total .npy files. Re-zipping to Drive...')
            !cd /content/SignSync/data && zip -q -r /content/drive/MyDrive/SignBridge/data/landmarks_new.zip landmarks/
            print('Checkpoint saved.')
            batch = []

print(f'\nDone. Saved: {saved}  Skipped: {skipped}  Errors: {errors}')

In [ ]:
# Cell 9 — Final zip and save to Drive
total = len(list(Path('/content/SignSync/data/landmarks').glob('*.npy')))
print(f'Total landmark files: {total}')
print('Zipping to Drive...')
!cd /content/SignSync/data && zip -q -r /content/drive/MyDrive/SignBridge/data/landmarks_new.zip landmarks/
print('Saved to /content/drive/MyDrive/SignBridge/data/landmarks_new.zip')
print('Rename this to landmarks.zip on Drive before running train_colab.ipynb')

In [ ]:
# Cell 10 — Rebuild labels_50.json based on new sample counts
# Keeps top 50 classes by sample count, remaps indices 0-49
import json
from pathlib import Path
from collections import Counter

landmarks_dir = Path('/content/SignSync/data/landmarks')

counts = Counter()
for f in landmarks_dir.glob('*.npy'):
    word = '_'.join(f.stem.split('_')[:-1])   # strip trailing _id
    counts[word] += 1

top50 = [word for word, _ in counts.most_common(50)]
new_label_map = {word: idx for idx, word in enumerate(top50)}

out_path = '/content/SignSync/data/labels_50.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(new_label_map, f, indent=2)

print('New labels_50.json:')
for word, idx in new_label_map.items():
    print(f'  {idx:2d}  {word:20s}  ({counts[word]} samples)')

# Save updated labels file to Drive
import shutil
shutil.copy(out_path, '/content/drive/MyDrive/SignBridge/data/labels_50.json')
print('\nSaved updated labels_50.json to Drive.')